In [0]:
dbutils.widgets.text("catalog_name","dbr_dev_ua_5816_trail_2")
dbutils.widgets.text("gold_schema","viktoriia_kalenichenko_gold")

In [0]:
CATALOG = dbutils.widgets.get("catalog_name")
GOLD_SCHEMA = dbutils.widgets.get("gold_schema")

GOLD = f"{CATALOG}.{GOLD_SCHEMA}"

DEMO = f"{GOLD}.governance_demo"
ROLES = f"{GOLD}.access_roles"
RLS_FUNCTION = f"{GOLD}.rls_subregion_filter"

In [0]:
spark.sql(f"""
CREATE OR REPLACE FUNCTION {RLS_FUNCTION}(
    row_subregion STRING
)
RETURNS BOOLEAN
RETURN EXISTS (
    SELECT 1
    FROM {ROLES}
    WHERE user_email = current_user()
      AND (
          role IN ('admin', 'analyst')
          OR (
              role = 'regional_viewer'
              AND allowed_subregion = row_subregion
          )
      )
)
""")

In [0]:
spark.sql(f"""
    ALTER TABLE {DEMO}
    SET ROW FILTER {RLS_FUNCTION} ON (subregion)
""")

In [0]:
CLS_FUNCTION = f"{GOLD}.mask_production"

spark.sql(f"""
CREATE OR REPLACE FUNCTION {CLS_FUNCTION}(
    production_value DOUBLE
)
RETURNS DOUBLE
RETURN CASE
    WHEN EXISTS (
        SELECT 1
        FROM {ROLES}
        WHERE user_email = current_user()
          AND role = 'admin'
    )
    THEN production_value
    ELSE CAST(NULL AS DOUBLE)
END
""")


In [0]:
spark.sql(f"""
    ALTER TABLE {DEMO}
    ALTER COLUMN total_production
    SET MASK {CLS_FUNCTION}
""")

In [0]:
display(
    spark.sql(f"""
        SELECT
            country_name,
            subregion,
            year,
            total_production
        FROM {DEMO}
        ORDER BY country_name, year
        LIMIT 10
    """)
)